# MiRAGeNews preprocessing

Prepares the arrays read by `notebooks/01_MiRAGeNews_FedSVLM.ipynb`.

1. **Encode.** Load MiRAGeNews from Hugging Face (`anson-huang/mirage-news`, pinned revision) and apply the CLIP ViT-B/32 processor exactly as in the Light-MFND repository (text truncated/padded to 77 tokens, images resized to 224×224 and CLIP-normalised). The encoded dataset is saved to Drive (`fed/mirage_clip_enc`).
2. **Convert to memory-mapped arrays.** Write `<split>_pix.npy` (float16), `<split>_ids.npy`, `<split>_msk.npy` and `<split>_lab.npy` to `fed/npy`, so that training does not hold the images in RAM. Run once; a CPU runtime is enough.

Split sizes: train 10,000; validation 2,500; five test sets of 500 each (`test1_nyt_mj` in-domain; `test2_bbc_dalle`, `test3_cnn_dalle`, `test4_bbc_sdxl`, `test5_cnn_sdxl` out-of-domain).

*Note: step 1 shows the output of its run on 16 Nov 2026. Step 2 was run on 17 Nov 2026 in a separate scratch session whose output was not saved; the code is the one used. Code comments are in Vietnamese.*

## Step 1 — Encode MiRAGeNews with the CLIP processor

In [ ]:
import os
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
from datasets import load_dataset, load_from_disk
from transformers import CLIPProcessor

FED = "/content/drive/MyDrive/NLP_project/fed"; os.makedirs(FED, exist_ok=True)
ENC = f"{FED}/mirage_clip_enc"
MODEL_NAME = "openai/clip-vit-base-patch32"

if os.path.exists(ENC):
    enc = load_from_disk(ENC)
else:
    ds = load_dataset("anson-huang/mirage-news", revision="b5a7e734850b4ec623ddee018a1d9e097fe248ef")
    processor = CLIPProcessor.from_pretrained(MODEL_NAME)
    def preprocess(ex):
        x = processor(text=ex["text"], images=ex["image"], truncation=True,
                      padding="max_length", max_length=77, return_tensors="pt")
        return {"input_ids": x["input_ids"][0], "attention_mask": x["attention_mask"][0],
                "pixel_values": x["pixel_values"][0], "labels": int(ex["label"])}
    enc = ds.map(preprocess, remove_columns=ds["train"].column_names)
    enc.save_to_disk(ENC)
enc.set_format("torch")
TESTS = [s for s in enc if s.startswith("test")]
print({s: len(enc[s]) for s in enc}); print("test sets:", TESTS)

Mounted at /content/drive


{'train': 10000, 'validation': 2500, 'test1_nyt_mj': 500, 'test2_bbc_dalle': 500, 'test3_cnn_dalle': 500, 'test4_bbc_sdxl': 500, 'test5_cnn_sdxl': 500}
test sets: ['test1_nyt_mj', 'test2_bbc_dalle', 'test3_cnn_dalle', 'test4_bbc_sdxl', 'test5_cnn_sdxl']


## Step 2 — Convert to memory-mapped `.npy` arrays

In [ ]:
# ===== CHẠY MỘT LẦN, nên dùng runtime CPU để đỡ tốn hạn mức GPU =====
import os, time, shutil, numpy as np
from numpy.lib.format import open_memmap
from datasets import load_from_disk
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
FED = "/content/drive/MyDrive/NLP_project/fed"; ENC = f"{FED}/mirage_clip_enc"
OUT = "/content/npy"; os.makedirs(OUT, exist_ok=True)
src = load_from_disk(ENC).with_format("numpy")
for sp in src:
    d = src[sp]; N = len(d); t0 = time.time()
    pix = open_memmap(f"{OUT}/{sp}_pix.npy", mode="w+", dtype=np.float16, shape=(N, 3, 224, 224))
    ids = np.zeros((N, 77), np.int64); msk = np.zeros((N, 77), np.int64); lab = np.zeros(N, np.int64)
    for i in range(0, N, 128):
        c = d[i:i + 128]; j = i + len(c["labels"])
        pix[i:j] = np.asarray(c["pixel_values"]).astype(np.float16)
        ids[i:j] = np.asarray(c["input_ids"]); msk[i:j] = np.asarray(c["attention_mask"]); lab[i:j] = np.asarray(c["labels"])
        if i == 0: print(f"{sp}: chunk đầu (128 mẫu) mất {time.time()-t0:.1f}s")
    pix.flush(); del pix
    np.save(f"{OUT}/{sp}_ids.npy", ids); np.save(f"{OUT}/{sp}_msk.npy", msk); np.save(f"{OUT}/{sp}_lab.npy", lab)
    print(f"{sp}: xong {N} mẫu trong {time.time()-t0:.0f}s")
shutil.copytree(OUT, f"{FED}/npy", dirs_exist_ok=True); print("Đã lưu lên Drive:", f"{FED}/npy")
